In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report
from sklearn.metrics import mean_squared_error

In [ ]:
data = pd.read_csv(
    "/content/drive/MyDrive/Colab/loan project/new_loan.csv",
    sep=None,
    engine="python"
)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048551 entries, 0 to 1048550
Columns: 148 entries, ﻿id to settlement_term
dtypes: float64(111), int64(3), object(34)
memory usage: 1.2+ GB


In [ ]:
data["issue_d"].value_counts(dropna=False)

,count
issue_d,
16-Mar,60460
15-Oct,48628
18-Aug,46071
15-Jul,45956
15-Dec,44336
17-Aug,43571
18-Jul,43086
17-Sep,39712
17-Jul,39412


In [ ]:
data.head()

,﻿id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600,3600,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700,24700,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000,20000,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000,35000,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400,10400,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
data['home_ownership'].value_counts(dropna=False)

,count
home_ownership,
MORTGAGE,512988
RENT,414154
OWN,121161
ANY,156
NaN,89
NONE,3


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048551 entries, 0 to 1048550
Columns: 148 entries, ﻿id to settlement_term
dtypes: float64(111), int64(3), object(34)
memory usage: 1.2+ GB


In [ ]:
data[["issue_year", "issue_month"]] = data["issue_d"].str.split("-", expand=True)

In [ ]:
data["issue_year"].value_counts()

,count
issue_year,
15,421046
18,236033
17,228137
16,163246


In [ ]:
data["loan_status"].value_counts(normalize=True)

,proportion
loan_status,
Fully Paid,0.460239
Current,0.400125
Charged Off,0.123204
Late (31-120 days),0.010375
In Grace Period,0.004031
Late (16-30 days),0.002006
Default,0.000020


In [ ]:
data["loan_status"] = data["loan_status"].replace({
    "Default": "Charged Off"
})

In [ ]:
data = data[data["loan_status"].isin(["Fully Paid", "Charged Off"])]

In [ ]:
data.reset_index(drop=True, inplace=True)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 611739 entries, 0 to 611738
Columns: 150 entries, ﻿id to issue_month
dtypes: float64(111), int64(3), object(36)
memory usage: 700.1+ MB


In [ ]:
data["emp_title"].value_counts(dropna=True)

,count
emp_title,
Teacher,11118
Manager,10233
Owner,5820
Registered Nurse,4501
RN,4461
...,...
"Owner, Fee Appraiser",1
User Researcher,1
IT Supply Chain Analyst,1


In [ ]:
data2=data[data['issue_year'].isin(['17','18'])]

In [ ]:
data2["emp_title"].value_counts()

,count
emp_title,
Manager,2151
Teacher,2096
Owner,1254
Driver,1026
Sales,889
...,...
Personal Vacation Advisor,1
Secretarial Assistant,1
Sheriffs technician,1


In [ ]:
data2=data2.drop(columns=["member_id","funded_amnt_inv","grade","desc","fico_range_low","fico_range_high","out_prncp_inv","total_pymnt_inv","last_fico_range_low","last_fico_range_high","policy_code"])

In [ ]:
data2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 121175 entries, 375500 to 559617
Columns: 140 entries, ﻿id to last_fico_score
dtypes: float64(108), int64(3), object(29)
memory usage: 130.4+ MB


In [ ]:
hardship_cols = [col for col in data2.columns if col.startswith("hardship")]

data2.drop(columns=hardship_cols, inplace=True)

/tmp/ipykernel_233/409631070.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2.drop(columns=hardship_cols, inplace=True)


In [ ]:
data2["fico_score"] = (data["fico_range_low"] + data["fico_range_high"]) / 2

/tmp/ipykernel_233/2065280809.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["fico_score"] = (data["fico_range_low"] + data["fico_range_high"]) / 2


In [ ]:
data2["last_fico_score"]=(data["last_fico_range_low"]+data["last_fico_range_high"])/2

/tmp/ipykernel_233/4197085187.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["last_fico_score"]=(data["last_fico_range_low"]+data["last_fico_range_high"])/2


In [ ]:
data2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 121175 entries, 375500 to 559617
Columns: 140 entries, ﻿id to last_fico_score
dtypes: float64(108), int64(3), object(29)
memory usage: 130.4+ MB


In [ ]:
data2["loan_status"].value_counts(normalize=True)

,proportion
loan_status,
Fully Paid,0.781366
Charged Off,0.218634


In [ ]:
data2=pd.read_csv("/content/drive/MyDrive/Colab/loan project/loan2.csv",sep=",")

/tmp/ipykernel_233/2485480158.py:1: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  data2=pd.read_csv("/content/drive/MyDrive/Colab/loan project/loan2.csv",sep=",")


In [ ]:
data2

,id,loan_amnt,funded_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,...,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,issue_year,issue_month,fico_score,last_fico_score
0,130956066,3000,3000,36 months,7.34,93.10,A4,9 years,RENT,52000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,18,Mar,762.0,762.0
1,130968727,5000,5000,36 months,11.98,166.03,B5,10+ years,OWN,55000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,18,Mar,677.0,677.0
2,130910225,7000,7000,36 months,11.98,232.44,B5,< 1 year,MORTGAGE,40000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,18,Mar,697.0,642.0
3,130966492,30000,30000,36 months,21.85,1143.39,D5,10+ years,OWN,57000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,18,Mar,682.0,697.0
4,130942737,21000,21000,60 months,20.39,560.94,D4,10+ years,OWN,85000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,18,Mar,667.0,657.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121170,102395624,35000,35000,60 months,13.99,814.21,C3,< 1 year,MORTGAGE,100000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,17,Apr,687.0,772.0
121171,102426796,12000,12000,60 months,28.69,378.65,F1,10+ years,MORTGAGE,64500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,17,Apr,662.0,712.0
121172,102556443,24000,24000,60 months,23.99,690.30,E2,< 1 year,RENT,107000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,17,Apr,672.0,249.5
121173,102196576,6000,6000,36 months,11.44,197.69,B4,5 years,RENT,41000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,17,Apr,672.0,717.0


In [ ]:
data2['issue_d']

,issue_d
375500,18-Mar
375501,18-Mar
375502,18-Mar
375503,18-Mar
375504,18-Mar
...,...
559613,17-Apr
559614,17-Apr
559615,17-Apr
559616,17-Apr


In [ ]:
data2["issue_year"] = data2["issue_year"].astype(int)
data2["issue_year"] = data2["issue_year"] + 2000

/tmp/ipykernel_233/514858314.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["issue_year"] = data2["issue_year"].astype(int)
/tmp/ipykernel_233/514858314.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["issue_year"] = data2["issue_year"] + 2000


In [ ]:
data2["issue_year"]

,issue_year
375500,2018
375501,2018
375502,2018
375503,2018
375504,2018
...,...
559613,2017
559614,2017
559615,2017
559616,2017


In [ ]:
data2["issue_month"] = pd.to_datetime(
    data2["issue_month"], format="%b"
).dt.month

/tmp/ipykernel_233/1318267690.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["issue_month"] = pd.to_datetime(


In [ ]:
data2["issue_month"]

,issue_month
375500,3
375501,3
375502,3
375503,3
375504,3
...,...
559613,4
559614,4
559615,4
559616,4


In [ ]:
data2["earliest_cr_line"]

,earliest_cr_line
375500,Jan-98
375501,1-Aug
375502,7-Mar
375503,Apr-00
375504,8-Nov
...,...
559613,May-96
559614,5-Nov
559615,Apr-95
559616,May-90


In [ ]:
data2["earliest_cr_line"] = data2["earliest_cr_line"].astype(str).str.strip()

/tmp/ipykernel_233/3952452924.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["earliest_cr_line"] = data2["earliest_cr_line"].astype(str).str.strip()


In [ ]:
parts = data2["earliest_cr_line"].str.split("-", expand=True)

In [ ]:
parts = data2["earliest_cr_line"].str.split("-", expand=True)

# Detect if first part is alphabet (Month-Year)
is_month_first = parts[0].str.isalpha()

# Case 1: Month-Year (Jan-98)
date1 = pd.to_datetime(
    data2["earliest_cr_line"],
    format="%b-%y",
    errors="coerce"
)

# Case 2: Year-Month (1-Aug means 2001-Aug)
year_fixed = parts[0].str.zfill(2)  # make 1 → 01
reconstructed = year_fixed + "-" + parts[1]

date2 = pd.to_datetime(
    reconstructed,
    format="%y-%b",
    errors="coerce"
)

# Combine properly
data2["earliest_cr_line_clean"] = np.where(
    is_month_first,
    date1,
    date2
)

data2["earliest_cr_line_clean"] = pd.to_datetime(
    data2["earliest_cr_line_clean"]
)

/tmp/ipykernel_233/1167418230.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["earliest_cr_line_clean"] = np.where(
/tmp/ipykernel_233/1167418230.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["earliest_cr_line_clean"] = pd.to_datetime(


In [ ]:
data2["earliest_cr_line_clean"]
data2["earliest_cr_line_clean"] = pd.to_datetime(
    data2["earliest_cr_line_clean"]
)

,earliest_cr_line_clean
375500,1998-01-01
375501,2001-08-01
375502,2007-03-01
375503,2000-04-01
375504,2008-11-01
...,...
559613,1996-05-01
559614,2005-11-01
559615,1995-04-01
559616,1990-05-01


In [ ]:
data2["earliest_cr_line_year"] = data2["earliest_cr_line_clean"].dt.year
data2["earliest_cr_line_month"] = data2["earliest_cr_line_clean"].dt.month

/tmp/ipykernel_233/3446019358.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["earliest_cr_line_year"] = data2["earliest_cr_line_clean"].dt.year
/tmp/ipykernel_233/3446019358.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["earliest_cr_line_month"] = data2["earliest_cr_line_clean"].dt.month


In [ ]:
data2["earliest_cr_line_year"]

,earliest_cr_line_year
375500,1998
375501,2001
375502,2007
375503,2000
375504,2008
...,...
559613,1996
559614,2005
559615,1995
559616,1990


In [ ]:

data2["emp_length"] = data2["emp_length"].astype(str)

# Replace special cases
data2["emp_length"] = data2["emp_length"].replace({
    "< 1 year": "0",
    "10+ years": "10"
})

# Remove text like " years" and " year"
data2["emp_length"] = data2["emp_length"].str.replace(" years", "", regex=False)
data2["emp_length"] = data2["emp_length"].str.replace(" year", "", regex=False)

# Convert to numeric
data2["emp_length"] = pd.to_numeric(data2["emp_length"], errors="coerce")

/tmp/ipykernel_233/356178978.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["emp_length"] = data2["emp_length"].astype(str)
/tmp/ipykernel_233/356178978.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2["emp_length"] = data2["emp_length"].replace({
/tmp/ipykernel_233/356178978.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata

In [ ]:
data2["emp_length"].isna().sum()

np.int64(8953)

In [ ]:
data2["emp_length"].dtype

dtype('float64')

In [ ]:
col={"sec_app_earliest_cr_line","funded_amnt","out_prncp","total_pymnt","total_rec_prncp","total_rec_int","total_rec_late_fee",
     "recoveries","earliest_cr_line","last_pymnt_d","last_pymnt_amnt","next_pymnt_d","last_credit_pull_d","settlement_status",
     "settlement_date","settlement_amount","settlement_percentage","settlement_term","collection_recovery_fee","payment_plan_start_date"}

In [ ]:
data2["issue_d"]

,issue_d
375500,18-Mar
375501,18-Mar
375502,18-Mar
375503,18-Mar
375504,18-Mar
...,...
559613,17-Apr
559614,17-Apr
559615,17-Apr
559616,17-Apr


In [ ]:
print(data2["issue_d"].head())

KeyError: 'issue_d'

In [ ]:
data2["credit_history"]

KeyError: 'credit_history'

In [ ]:
data2["credit_history_years"] = (
    (data2["issue_d"] - data2["earliest_cr_line_clean"])
    .dt.days / 365.25
)

TypeError: cannot subtract DatetimeArray from ndarray

In [ ]:
data2["credit_history_years"]=data2["credit_history_years"].astype(int)

In [ ]:
data2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 121175 entries, 375500 to 559617
Columns: 133 entries, ﻿id to credit_history_years
dtypes: datetime64[ns](2), float64(100), int32(3), int64(5), object(23)
memory usage: 122.5+ MB


In [ ]:
data2.to_csv("/content/drive/MyDrive/Colab/loan project/loan3.csv", index=False)

In [ ]:
data4=pd.read_csv("/content/drive/MyDrive/Colab/loan project/loan4.csv",sep=",",engine="python")

In [ ]:
data3["loan_status"]=data3["loan_status"].replace({"Fully Paid":0,"Charged Off":1})

/tmp/ipykernel_186/712816700.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data3["loan_status"]=data3["loan_status"].replace({"Fully Paid":0,"Charged Off":1})


In [ ]:
data3["loan_status"]

NameError: name 'data3' is not defined

In [ ]:
data3["initial_list_status"]=data3["initial_list_status"].replace({"w":"whole","f":"fracional"})

In [ ]:
data4.to_csv("/content/drive/MyDrive/Colab/loan project/loan5.csv", sep=",", index=False)

NameError: name 'data4' is not defined

In [ ]:
data4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121175 entries, 0 to 121174
Columns: 133 entries, id to credit_history_years
dtypes: float64(100), int64(9), object(24)
memory usage: 123.0+ MB


In [ ]:
col={"funded_amnt","issue_d","pymnt_plan","earliest_cr_line","last_pymnt_d","last_credit_pull_d","next_pymnt_d","sec_app_earliest_cr_line","deferral_term",
     "payment_plan_start_date","orig_projected_additional_accrued_interest","debt_settlement_flag_date","settlement_date","earliest_cr_line_clean"}
data4 = data4.drop(columns=col)

In [ ]:
data4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121175 entries, 0 to 121174
Columns: 119 entries, id to credit_history_years
dtypes: float64(98), int64(8), object(13)
memory usage: 110.0+ MB


In [ ]:
data4.to_excel("/content/drive/MyDrive/Colab/loan project/loan5.xlsx", index=False)